## Assignment 4: Python translation

This notebook translates the R BMI summary in `polyglot-parity.qmd` to Python/pandas, one step per cell. After each step, an audit note compares the output with R.

The data path starts from this notebook's folder, because Jupyter runs from the notebook's location. R uses a repo-root path because `_quarto.yml` sets `execute-dir: project`. Both point to the same file.

In [ ]:
import pandas as pd
from helpers import load_and_preview

DATA_PATH = "../../../examples/nhanes-equity/data/nhanes_equity_v6.csv"

preview = load_and_preview(DATA_PATH)
preview

Audit check: the helper reports 105,626 rows and 18 columns, matching `nrow(nhanes)` and `ncol(nhanes)` in R. The `DtypeWarning` about `Marital` and `MaritalClean` comes from pandas guessing column types in chunks: these 
columns are empty in some parts of the file and text in others. Neither column is used in this analysis.

In [ ]:
nhanes = pd.read_csv(DATA_PATH, low_memory=False)
nhanes.shape

Audit check: the full load gives (105626, 18), the same shape as the helper and R. `low_memory=False` makes pandas read the whole file before guessing column types, so the warning does not repeat. It does not change BMI, IncomeGroup 
or Gender.

In [ ]:
nhanes.columns.tolist()

Audit check: the same 18 column names, in the same order, as `names(nhanes)` in R.

In [ ]:
nhanes["BMI"].isna().sum()

Audit check: 9,356 missing BMI values, matching `sum(is.na(nhanes$BMI))` in R. Both pandas and readr read the text `NA` in the CSV as missing, so missingness is handled the same way.

## Translate the R pipeline, one step at a time

In [ ]:
filtered = nhanes.dropna(subset=["BMI", "IncomeGroup", "Gender"])
len(filtered)

Audit check: 87,647 rows remain, matching the filtered row count in R. `dropna(subset=...)` drops a row if any of the three columns is missing, the same rule as `filter(!is.na(BMI), !is.na(IncomeGroup), !is.na(Gender))`.

In [ ]:
py_summary = (
    filtered
    .groupby(["IncomeGroup", "Gender"], as_index=False)
    .agg(
        n=("BMI", "size"),
        mean_bmi=("BMI", "mean"),
        sd_bmi=("BMI", "std"),
    )
)
py_summary

Audit check: 6 rows with the same income group and gender labels and the same counts (n) as `r_summary`. pandas' `"std"` divides by n − 1 by default, the same as R's `sd()`. Values are not rounded yet.

In [ ]:
py_summary["mean_bmi"] = py_summary["mean_bmi"].round(2)
py_summary["sd_bmi"] = py_summary["sd_bmi"].round(2)
py_summary

Audit check: after rounding to 2 decimals, the means and standard deviations match the stored R values in `r_summary$mean_bmi` and `r_summary$sd_bmi`. pandas shows 8.20 where the R tibble printed 8.2; the value is the same.

In [ ]:
py_summary = py_summary.sort_values(["IncomeGroup", "Gender"]).reset_index(drop=True)
py_summary

Audit check: rows are in the same order as `r_summary`. `groupby()` already sorts the groups, so this step changes nothing here, but it mirrors `arrange()` explicitly.

In [ ]:
print(len(filtered))
print(len(py_summary))
print(py_summary["mean_bmi"].tolist())
print(py_summary["sd_bmi"].tolist())

Audit check: these values match the R checks in `polyglot-parity.qmd` line for line. They inform the parity table.

## AI use

I used the GitHub Copilot Chat in Codespaces to translate the R pipeline to pandas one step at a time, with this prompt: *"Translate this R dplyr pipeline into Python/pandas one step at a time. For each step, explain the equivalent Python operation, the output I should inspect, and the parity check I should run against my R result. Do not combine steps or add new analysis choices."*

I then used Claude (Opus 5.5) to review and simplify Copilot's draft. Changes:

- Filter: replaced `.loc[...]` with three `.notna()` conditions with `dropna(subset=[...])`. It applies the same rule and matches the textbook's parity check command.
- Standard deviation: replaced a `lambda` using `std(ddof=1)` with `"std"`, because pandas already divides by n − 1 by default.
- Grouping: removed `sort=False`, so pandas sorts the groups the same way R's `group_by()` does.
- Removed extras the Python side does not need: a mean age column, and Python versions of the R function and the Waist and Race exercises.
- Kept `low_memory=False` from the draft load, to avoid a column-type warning in columns this analysis does not use.

Claude also drafted the wording of the audit notes. I verified each step by checking row counts, group labels and rounded values against the R output, as recorded in the audit notes above.